# One Conservative Q-Learning value update
**Team 12 — CQL MiniGrid.** This notebook uses the loss functions from the project itself. It runs on CPU, requires no Git or internet after installation, and does not train PPO or a full offline agent.

## 1. Project overview
The agent navigates `MiniGrid-Dynamic-Obstacles-8x8-v0`, reaching a goal while avoiding moving obstacles and walls. Its actions are **left (0), right (1), and forward (2)**. Episodes last at most 256 steps. Real task rewards are 0 for an ordinary step, −1 for a collision, and $1-0.9t/256$ for reaching the goal at step $t$.

PPO supplies frozen experience: stronger-only data, an episode-sampled mixture of final PPO/early PPO/random policies, and a transition-balanced mixture. Offline DQN, CQL, and behavioral cloning learn from those recorded transitions. They do not collect new environment experience during their updates. CQL adds a penalty that discourages unsupported actions from receiving excessively high action values.

In [1]:
import json
import math
from pathlib import Path

import torch
from cql_minigrid.common import seed_everything
from cql_minigrid.offline import td_targets, q_learning_loss, conservative_penalty

# Jupyter normally starts in the notebook directory; the executor also sets this directory.
ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents]
            if (parent / "pyproject.toml").is_file()
            and (parent / "src/cql_minigrid/offline.py").is_file())
seed_everything(0)
print("Imported the project's TD target and CQL loss functions; computation uses CPU.")

Imported the project's TD target and CQL loss functions; computation uses CPU.


## 2. DQN and discrete CQL
DQN fits $Q(s,a)$ to a Bellman target computed with a separate target network. Discrete CQL uses the same TD term and adds a conservative penalty over all available actions.

| Symbol | Meaning |
|---|---|
| $s$, $s'$ | Current and next state |
| $a$ | Action recorded in the offline data |
| $r$ | Recorded reward |
| $\gamma$ | Discount factor |
| $Q(s,a)$ | Predicted action value |
| $Q_{\mathrm{target}}$ | Target network used for next-state values |
| $\alpha$ | Conservative penalty coefficient |
| $y$ | Bellman TD target |

For the project's observed finite horizon, $d=\texttt{terminated}\lor\texttt{truncated}$ and
$$y=r+\gamma(1-d)\max_{a'}Q_{\mathrm{target}}(s',a').$$
For a minibatch $B$, the exact implemented loss is
$$L_{\mathrm{CQL}}=\frac{1}{|B|}\sum_{i\in B}\left[\operatorname{Huber}_1(Q(s_i,a_i)-y_i)+\alpha\left(\log\sum_b e^{Q(s_i,b)}-Q(s_i,a_i)\right)\right].$$
Here $\operatorname{Huber}_1(e)=e^2/2$ for $|e|\leq1$, and $|e|-1/2$ otherwise. The implementation uses `smooth_l1_loss` with its default beta 1 and mean reduction, and stable `torch.logsumexp`. **The TD loss is Huber, not MSE.** DQN is the $\alpha=0$ case. Real neural-network training uses Adam and a target network.

## 3. One numerical CQL update
We use **one illustrative recorded transition** and three directly trainable Q-values instead of a neural network. The reward 1 in this **nonterminal toy transition** is chosen for transparent arithmetic; it is not claimed to be a logged MiniGrid transition (ordinary real steps have reward 0).

The current values are `[0.2, 0.6, -0.1]`; action 1 (**right**) was recorded. The frozen target-network values at the next state are `[0.4, 0.8, 0.3]`. We set $\gamma=0.9$ and $\alpha=1$. This example uses one SGD step with learning rate 0.1 so that the update can be inspected directly. The project uses a neural network and Adam, with **the same loss formula**.

In [2]:
initial_q = torch.tensor([[0.2, 0.6, -0.1]], dtype=torch.float32)
q_values = torch.nn.Parameter(initial_q.clone())
action = torch.tensor([1], dtype=torch.long)
reward = torch.tensor([1.0])
next_target_q = torch.tensor([[0.4, 0.8, 0.3]])
terminated = torch.tensor([False])
truncated = torch.tensor([False])
gamma, alpha, learning_rate = 0.9, 1.0, 0.1
optimizer = torch.optim.SGD([q_values], lr=learning_rate)

print("Action order: left, right, forward")
print("Initial Q-values:", initial_q.tolist()[0])
print("Recorded action:", action.item(), "(right)")

Action order: left, right, forward
Initial Q-values: [0.20000000298023224, 0.6000000238418579, -0.10000000149011612]
Recorded action: 1 (right)


In [3]:
with torch.no_grad():
    max_next_q = next_target_q.max(dim=1).values
    target = td_targets(reward, max_next_q, terminated, truncated, gamma)
predicted = q_values.gather(1, action[:, None]).squeeze(1)
print(f"Maximum target-network next Q: {max_next_q.item():.6f}")
print(f"TD target y = 1 + 0.9 * 0.8: {target.item():.6f}")
print(f"Predicted Q for recorded action: {predicted.item():.6f}")
assert math.isclose(target.item(), 1.72, abs_tol=1e-6)
assert not target.requires_grad

Maximum target-network next Q: 0.800000
TD target y = 1 + 0.9 * 0.8: 1.720000
Predicted Q for recorded action: 0.600000


The TD residual is $0.6-1.72=-1.12$. Its magnitude exceeds 1, so the linear branch of Huber gives $L_{\mathrm{TD}}=1.12-0.5=0.62$. The conservative term is $\log(e^{0.2}+e^{0.6}+e^{-0.1})-0.6$. For this single transition, it is also the negative log-softmax probability of the recorded action.

In [4]:
loss, td_loss, penalty = q_learning_loss(q_values, action, target, alpha)
logsumexp = torch.logsumexp(q_values, dim=1)
expected_logsumexp = math.log(sum(math.exp(value) for value in [0.2, 0.6, -0.1]))
print(f"Huber TD loss: {td_loss.item():.6f}")
print(f"logsumexp of current Q-values: {logsumexp.item():.6f}")
print(f"Conservative penalty: {penalty.item():.6f}")
print(f"Total CQL loss = TD + alpha * penalty: {loss.item():.6f}")
assert math.isclose(td_loss.item(), 0.62, abs_tol=1e-6)
assert math.isclose(penalty.item(), expected_logsumexp - 0.6, abs_tol=1e-6)
torch.testing.assert_close(penalty, conservative_penalty(q_values, action))
torch.testing.assert_close(loss, td_loss + alpha * penalty)

Huber TD loss: 0.620000
logsumexp of current Q-values: 1.373300
Conservative penalty: 0.773300
Total CQL loss = TD + alpha * penalty: 1.393300


For action $j$, the penalty gradient is $\operatorname{softmax}(Q)_j-\mathbf{1}[j=a]$. The TD gradient is −1 for the recorded action and zero for the other two values in this example. With $\alpha=1$, the total recorded-action gradient is therefore $\operatorname{softmax}(Q)_a-2$. Positive gradients on the two unrecorded actions make their values decrease under gradient descent.

In [5]:
optimizer.zero_grad(set_to_none=True)
loss.backward()
probabilities = [math.exp(value - expected_logsumexp) for value in [0.2, 0.6, -0.1]]
expected_gradient = torch.tensor([probabilities])
expected_gradient[0, action.item()] -= 2.0
print("Autograd gradients:", [round(v, 6) for v in q_values.grad.tolist()[0]])
torch.testing.assert_close(q_values.grad, expected_gradient, rtol=1e-6, atol=1e-6)
assert q_values.grad[0, 0] > 0 and q_values.grad[0, 2] > 0
assert q_values.grad[0, 1] < 0
before_update = q_values.detach().clone()
expected_after = before_update - learning_rate * q_values.grad.detach()
optimizer.step()
print("Q before:", [round(v, 6) for v in before_update.tolist()[0]])
print("Q after: ", [round(v, 6) for v in q_values.detach().tolist()[0]])
torch.testing.assert_close(q_values.detach(), expected_after, rtol=1e-6, atol=1e-6)
assert q_values[0, 1] > before_update[0, 1]
assert q_values[0, 0] < before_update[0, 0] and q_values[0, 2] < before_update[0, 2]

Autograd gradients: [0.309344, -1.538512, 0.229168]
Q before: [0.2, 0.6, -0.1]
Q after:  [0.169066, 0.753851, -0.122917]


## 4. One DQN update versus one CQL update
We restart from the same initial values for both updates. DQN adjusts only the recorded value in this direct-value example. CQL also lowers the unrecorded values and adds a gradient favoring the recorded action. With a shared neural network, changing parameters can affect several state/action values even for DQN; the simple three-parameter example isolates the loss's direct action-value gradients.

In [6]:
def one_update(coefficient):
    values = torch.nn.Parameter(initial_q.clone())
    stepper = torch.optim.SGD([values], lr=learning_rate)
    objective, _, _ = q_learning_loss(values, action, target, coefficient)
    stepper.zero_grad(set_to_none=True)
    objective.backward()
    gradient = values.grad.detach().clone()
    stepper.step()
    return values.detach().clone(), gradient

q_dqn, gradient_dqn = one_update(0.0)
q_cql, gradient_cql = one_update(1.0)
print(f"{'Action':<10} {'Initial':>10} {'DQN alpha=0':>14} {'CQL alpha=1':>14}")
for index, name in enumerate(['left', 'right', 'forward']):
    print(f"{name:<10} {initial_q[0,index].item():>10.6f} {q_dqn[0,index].item():>14.6f} {q_cql[0,index].item():>14.6f}")
torch.testing.assert_close(q_dqn, torch.tensor([[0.2, 0.7, -0.1]]), rtol=1e-6, atol=1e-6)
torch.testing.assert_close(gradient_dqn, torch.tensor([[0.0, -1.0, 0.0]]))
torch.testing.assert_close(q_cql, q_values.detach())
assert torch.equal(q_dqn[0, [0, 2]], initial_q[0, [0, 2]])
assert torch.all(q_cql[0, [0, 2]] < q_dqn[0, [0, 2]])
print("All numerical loss, gradient, and update assertions passed.")

Action        Initial    DQN alpha=0    CQL alpha=1
left         0.200000       0.200000       0.169066
right        0.600000       0.700000       0.753851
forward     -0.100000      -0.100000      -0.122917
All numerical loss, gradient, and update assertions passed.


## 5. Connection to the real implementation
Real observations contain **837 features**: the full symbolic 8×8 grid, direction, and normalized `elapsed_steps`. The Q-network has **two 128-unit hidden layers with ReLU** and three action outputs. Each update samples a minibatch from a frozen offline dataset. A separate target network generates TD targets; its parameters are copied from the online network at a fixed interval.

`train_offline` in `src/cql_minigrid/offline.py` uses **Adam**, clips gradient norm at 10, and never resets or steps an environment. This notebook's directly trainable values and single SGD step are an arithmetic demonstration, not full neural-network training. Evaluation uses held-out environment seeds, separated from PPO training, collection, and validation seeds. The real main-run settings are read below rather than inferred from the toy example.

In [7]:
main_config = json.loads((ROOT / 'configs/main.json').read_text())
observation = json.loads((ROOT / 'configs/observation.json').read_text())
settings = main_config['offline']
print("Observation features:", observation['shape'][0])
for key in ['hidden', 'updates', 'batch_size', 'learning_rate', 'gamma', 'target_interval', 'alpha']:
    print(f"{key}: {settings[key]}")
assert observation['shape'][0] == 837 and settings['hidden'] == [128, 128]

Observation features: 837
hidden: [128, 128]
updates: 5000
batch_size: 256
learning_rate: 0.0003
gamma: 0.99
target_interval: 250
alpha: 1.0


## 6. Actual project results
The following measurements come from the saved compact reports, not from the illustrative transition. No dataset loading or retraining is needed.

- **Offline DQN:** Huber TD fitting with no conservative penalty.
- **CQL:** the same TD loss plus the discrete conservative penalty.
- **Behavioral cloning (BC):** supervised cross-entropy fitting of recorded actions.

The primary experiment uses two approximately 200,000-transition datasets, three training seeds per method, 5,000 updates per run, and 100 common held-out test seeds. Values below are means and sample standard deviations across training seeds. Successful-step means condition on successful episodes; missing cases are shown as `—`.

In [8]:
primary = json.loads((ROOT / 'reports/summary.json').read_text())
study = json.loads((ROOT / 'reports/ablation/summary.json').read_text())

def stat(row, metric, scale=1, digits=1):
    value = row[metric]
    return '—' if value['mean'] is None else f"{value['mean']*scale:.{digits}f} ± {value['std']*scale:.{digits}f}"

print(f"Primary experiment: {len(primary['runs'])} runs; {len(primary['seeds']['evaluation'])} test seeds per run")
print(f"{'Dataset':<8} {'Method':<6} {'Success %':>14} {'Collision %':>14} {'Timeout %':>14} {'Return':>16} {'Success steps':>16}")
for row in primary['aggregate']:
    print(f"{row['dataset']:<8} {row['method'].upper():<6} {stat(row,'success_rate',100):>14} {stat(row,'collision_rate',100):>14} {stat(row,'timeout_rate',100):>14} {stat(row,'return_mean',digits=3):>16} {stat(row,'successful_steps_mean'):>16}")
print("Final PPO collector success on the same test seeds:", f"{primary['references']['final']['success_rate']*100:.1f}%")

Primary experiment: 18 runs; 100 test seeds per run
Dataset  Method      Success %    Collision %      Timeout %           Return    Success steps
mixed    BC         55.0 ± 4.6     19.0 ± 2.0     26.0 ± 5.6    0.248 ± 0.037       57.9 ± 3.4
mixed    CQL        60.7 ± 6.1     13.3 ± 1.2     26.0 ± 6.0    0.357 ± 0.039       54.2 ± 6.9
mixed    DQN       15.7 ± 26.3    28.0 ± 11.1    56.3 ± 36.8   -0.135 ± 0.146       21.1 ± 0.1
strong   BC         58.3 ± 5.0     22.3 ± 0.6     19.3 ± 5.1    0.239 ± 0.047       59.1 ± 3.6
strong   CQL        61.7 ± 1.5     21.3 ± 1.2     17.0 ± 2.6    0.276 ± 0.004       58.6 ± 0.6
strong   DQN         0.0 ± 0.0    100.0 ± 0.0      0.0 ± 0.0   -1.000 ± 0.000                —
Final PPO collector success on the same test seeds: 62.0%


The original mixed file chose sources per episode and consequently contained about 93.8% final-PPO transitions. The follow-up controls the transition shares at approximately 50% final PPO, 25% early PPO, and 25% random. It compares prespecified $\alpha\in\{0,0.1,1,5\}$ on all three files and includes BC on balanced data.

High alpha values substantially degraded CQL on the transition-balanced file. BC also failed there. The alpha-zero DQN control had large variation across its three seeds. A larger conservative penalty is therefore not uniformly beneficial when much recorded behavior is poor. The default alpha was not selected from these test results. There is one collector training seed and one frozen dataset per condition; data coverage changes together with source quality. Three training seeds do not establish statistical significance or general algorithm superiority.

In [9]:
print(f"Follow-up: {len(study['runs'])} runs; alpha values {study['config']['study']['alphas']}")
print(f"{'Dataset':<16} {'Method':<6} {'Alpha':>6} {'Success %':>14} {'Return':>16}")
for row in study['aggregate']:
    coefficient = '—' if row['alpha'] is None else f"{row['alpha']:g}"
    print(f"{row['dataset']:<16} {row['method'].upper():<6} {coefficient:>6} {stat(row,'success_rate',100):>14} {stat(row,'return_mean',digits=3):>16}")
zero_runs = sorted((r for r in study['runs'] if r['dataset']=='mixed_balanced' and r['method']=='cql' and r['alpha']==0), key=lambda r:r['seed'])
print("Balanced-data DQN success by training seed:", [(r['seed'], round(100*r['metrics']['success_rate'], 1)) for r in zero_runs])
assert primary['seeds']['evaluation'] == study['seeds']['evaluation']
print("Results were read from saved reports; no training or evaluation was run by this notebook.")

Follow-up: 39 runs; alpha values [0.0, 0.1, 1.0, 5.0]
Dataset          Method  Alpha      Success %           Return
mixed_balanced   BC          —      0.0 ± 0.0   -0.997 ± 0.006
mixed_balanced   CQL         0    61.0 ± 51.1    0.429 ± 0.574
mixed_balanced   CQL       0.1    48.0 ± 25.2    0.323 ± 0.163
mixed_balanced   CQL         1      2.7 ± 2.3   -0.939 ± 0.055
mixed_balanced   CQL         5      0.0 ± 0.0   -0.997 ± 0.006
mixed_episode    CQL         0    15.7 ± 26.3   -0.135 ± 0.146
mixed_episode    CQL       0.1    58.3 ± 10.0    0.331 ± 0.046
mixed_episode    CQL         1     60.7 ± 6.1    0.357 ± 0.039
mixed_episode    CQL         5     56.3 ± 4.2    0.287 ± 0.020
strong           CQL         0      0.0 ± 0.0   -1.000 ± 0.000
strong           CQL       0.1     61.3 ± 8.5    0.300 ± 0.062
strong           CQL         1     61.7 ± 1.5    0.276 ± 0.004
strong           CQL         5     56.7 ± 2.9    0.247 ± 0.010
Balanced-data DQN success by training seed: [(0, 91.0), (1, 90.0

**Reproduction:** see `SUBMISSION_README.md` for installation, fresh-kernel execution, tests, and evaluation of saved agents. `docs/ablation.md` and the JSON/CSV files in `reports/` contain the complete experiment details and limitations.